# 03 — Features: cómo 60.000 números se resumen en 105

**Objetivo:** entender qué mide cada característica (*feature*) que usan los
modelos. Al terminar deberías poder explicar:
- por qué los modelos necesitan un vector de largo fijo y cómo se obtiene
  (tramas + estadísticos);
- qué mide cada familia: energía (RMS), cruces por cero, centroide, ancho de
  banda, roll-off, planitud, contraste y MFCC;
- qué features dependen del volumen de la grabación;
- por qué la frecuencia de carga (22.050 vs. 4000 Hz) cambia lo que ven los MFCC.

Usamos la función de extracción de la tesis (`src/features.py`), así lo que
veas acá es exactamente lo que reciben los modelos.

In [ ]:
import sys
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.features import extract_features
from src.paths import AUDIO, DATA_DIR, OUT_DIR

SR_TESIS = 22050   # frecuencia a la que carga src/features.py
SEG = 4            # segundos que mostramos en los gráficos

hs = pd.read_csv(DATA_DIR / "HS.csv")
EJEMPLOS = {"Normal": "F_N_RC",
            "Soplo": hs.loc[hs["Heart Sound Type"] == "Mid Systolic Murmur", "Heart Sound ID"].iloc[0]}


def cargar(file_id, subset="HS", sr=SR_TESIS):
    y, _ = librosa.load(AUDIO[subset] / f"{file_id}.wav", sr=sr)
    return y

## 1. El problema: audios → tabla

Los modelos clásicos (regresión logística, random forest, etc.) esperan una
**tabla**: una fila por grabación y una columna por característica, siempre
las mismas columnas. Un audio son decenas de miles de números que además
cambian con el tiempo. La solución tiene dos pasos:

1. **Tramas:** se corta el audio en ventanas cortas y en cada una se calcula
   una medida (por ejemplo, la energía). Resultado: una *serie temporal* de
   esa medida.
2. **Estadísticos:** esa serie se resume con pocos números (media, desvío,
   máximo, mínimo). Resultado: un **vector de largo fijo**, sin importar
   cuánto dure la grabación.

Es lo que hace el clínico al auscultar: resume un sonido continuo en unos
pocos descriptores (intensidad, tono, en qué momento del ciclo aparece).

Parámetros de las tramas (valores por defecto de librosa, los que usa la tesis):
- `n_fft = 2048` muestras → a 22.050 Hz son **93 ms** por trama;
- `hop_length = 512` muestras → una trama nueva cada **23 ms**.

In [ ]:
y = cargar(EJEMPLOS["Normal"])
rms_tramas = librosa.feature.rms(y=y)[0]
t_tramas = librosa.frames_to_time(np.arange(len(rms_tramas)), sr=SR_TESIS)
print(f"Muestras del audio: {len(y)}  ->  tramas: {len(rms_tramas)}  ->  4 números: "
      f"media={rms_tramas.mean():.4f}, desvío={rms_tramas.std():.4f}, "
      f"máx={rms_tramas.max():.4f}, mín={rms_tramas.min():.4f}")

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
t = np.arange(len(y)) / SR_TESIS
axes[0].plot(t, y, lw=0.4)
axes[0].set(ylabel="Amplitud", title="Paso 0: el audio (miles de muestras)", xlim=(0, SEG))
axes[1].plot(t_tramas, rms_tramas, "o-", ms=3)
axes[1].axhline(rms_tramas.mean(), color="red", ls="--", label="media")
axes[1].set(xlabel="Tiempo (s)", ylabel="RMS por trama",
            title="Paso 1: una medida por trama  ·  Paso 2: se resume (la línea roja es la media)")
axes[1].legend()
fig.tight_layout()

Ojo con lo que se pierde en el paso 2: la media y el desvío de la energía son
iguales si los latidos son regulares o irregulares, siempre que sean los
mismos latidos en otro orden. **El orden temporal desaparece.** Por eso una
arritmia puede quedar mal representada (sección 2.7.3 de la tesis).

## 2. Energía: RMS

**RMS** (*root mean square*) = raíz del promedio de los cuadrados de las
muestras de cada trama. Mide la **intensidad**: cuánto se mueve la señal.
La tesis guarda `rms_mean`, `rms_std` y `rms_max`.

Depende directamente del volumen de la grabación (notebook 02, sección 6).

## 3. Tasa de cruces por cero (ZCR)

En cada trama, la fracción de muestras en las que la señal **cambia de
signo**. Una señal grave oscila lento y cruza el cero pocas veces; una aguda
o ruidosa lo cruza muchas veces. Es una medida rápida y burda de "cuán aguda
o ruidosa" es la señal. Es sensible al siseo de fondo, aunque sea débil.

## 4. Las features espectrales

Se calculan sobre el **espectro de cada trama** (la misma información que una
columna del espectrograma):

| Feature | Qué mide | Intuición |
|---|---|---|
| **Centroide** | el "centro de masa" del espectro | qué tan "brillante" (agudo) suena |
| **Ancho de banda** | cuánto se dispersan las frecuencias alrededor del centroide | sonido concentrado vs. desparramado |
| **Roll-off** | la frecuencia debajo de la cual está el 85 % de la energía | dónde "termina" el sonido |
| **Planitud** (*flatness*) | qué tan plano es el espectro (0 = tono puro, 1 = ruido blanco) | ¿tonal o ruidoso? |
| **Contraste** | diferencia entre picos y valles del espectro, en 7 bandas | ¿hay componentes tonales que sobresalen? |

Comparamos cómo evolucionan en el tiempo para un normal y un soplo:

In [ ]:
def series(y, sr=SR_TESIS):
    return {"RMS": librosa.feature.rms(y=y)[0],
            "ZCR": librosa.feature.zero_crossing_rate(y)[0],
            "Centroide (Hz)": librosa.feature.spectral_centroid(y=y, sr=sr)[0],
            "Ancho de banda (Hz)": librosa.feature.spectral_bandwidth(y=y, sr=sr)[0],
            "Roll-off 85 % (Hz)": librosa.feature.spectral_rolloff(y=y, sr=sr)[0],
            "Planitud": librosa.feature.spectral_flatness(y=y)[0]}


fig, axes = plt.subplots(6, 1, figsize=(11, 13), sharex=True)
for nombre, file_id in EJEMPLOS.items():
    s = series(cargar(file_id))
    tt = librosa.frames_to_time(np.arange(len(s["RMS"])), sr=SR_TESIS)
    for ax, (feat, valores) in zip(axes, s.items()):
        ax.plot(tt, valores, lw=1, label=f"{nombre} ({file_id})")
        ax.set(ylabel=feat)
axes[0].legend()
axes[-1].set(xlabel="Tiempo (s)", xlim=(0, SEG))
fig.tight_layout()

Preguntas para mirar el gráfico:
- ¿En qué momentos sube el centroide? ¿Coincide con los ruidos o con los
  silencios?
- ¿El soplo tiene la planitud más alta o más baja que el normal? Pensá en que
  el soplo es "similar a un ruido" (sección 2.2 de la tesis).
- Fijate la escala del centroide y del roll-off: ¿llegan cerca de 2000 Hz,
  donde no hay sonido? (Recordá la notebook 01.)

## 5. MFCC: el resumen de la "forma" del espectro

Los MFCC (*coeficientes cepstrales en frecuencia mel*) se calculan en cada trama así:
1. se toma el espectro de la trama;
2. se lo pasa por un **banco de filtros mel**: bandas angostas en los graves y
   anchas en los agudos, que imitan cómo percibe el oído;
3. se toma el logaritmo de la energía de cada banda (como los dB);
4. se aplica una transformada (DCT) que resume esas energías en pocos
   coeficientes.

Cómo leerlos:
- **MFCC 1:** la energía total de la trama, en escala logarítmica. Es,
  básicamente, **el volumen**.
- **MFCC 2:** la "inclinación" del espectro: si hay más energía en graves o en agudos.
- **MFCC 3 en adelante:** detalles cada vez más finos de la forma del espectro.

Los **delta-MFCC** miden cuánto cambia cada coeficiente de una trama a la
siguiente: la "velocidad" de cambio del sonido.

### El banco de filtros mel depende de la frecuencia de carga

librosa reparte 128 bandas mel entre 0 Hz y la mitad de la frecuencia de
carga. Comparamos cargar a 22.050 Hz (lo que hace la tesis) con la frecuencia
nativa de 4000 Hz:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
for ax, sr in zip(axes, [22050, 4000]):
    banco = librosa.filters.mel(sr=sr, n_fft=2048, n_mels=128)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
    for fila in banco[::2]:
        ax.plot(freqs, fila / fila.max(), lw=0.6)
    centros = librosa.mel_frequencies(n_mels=130, fmin=0, fmax=sr / 2)[1:-1]
    ax.axvspan(2000, sr / 2, color="gray", alpha=0.3, label="zona sin sonido (> 2000 Hz)")
    ax.axvspan(0, 400, color="red", alpha=0.15, label="banda cardíaca (< 400 Hz)")
    ax.set(xlim=(0, sr / 2), title=f"Cargando a {sr} Hz: {(centros < 400).sum()} de 128 bandas debajo de 400 Hz, "
                                  f"{(centros > 2000).sum()} en la zona vacía", yticks=[])
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("Frecuencia (Hz)")
fig.tight_layout()

A 22.050 Hz, **la mitad de las bandas** miran una zona donde no hay sonido, y
la banda cardíaca queda cubierta por pocas. A 4000 Hz todas las bandas caen
donde hay señal y la banda cardíaca tiene el doble de resolución.

**Cuidado si cambiamos a 4000 Hz:** `n_fft = 2048` muestras a 4000 Hz son
512 ms por trama, más que un ciclo cardíaco completo (y R1 y R2 quedarían en
la misma trama). Habría que achicarla, por ejemplo a 256 muestras (64 ms).
Los parámetros de las tramas están pensados en **tiempo**, no en muestras.

Así se ven los MFCC de los dos ejemplos (configuración de la tesis):

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, (nombre, file_id) in zip(axes, EJEMPLOS.items()):
    mfcc = librosa.feature.mfcc(y=cargar(file_id), sr=SR_TESIS, n_mfcc=13)
    img = librosa.display.specshow(mfcc, sr=SR_TESIS, x_axis="time", ax=ax)
    ax.set(title=f"MFCC — {nombre} ({file_id})", ylabel="Coeficiente", xlim=(0, SEG))
    ax.set_yticks(np.arange(13) + 0.5, labels=np.arange(1, 14))
fig.colorbar(img, ax=axes)

Cada fila es un coeficiente y cada columna una trama. La fila 1 (abajo) sigue
a los latidos porque es la energía: oscura en los silencios, clara en los ruidos.

## 6. Las 105 features de una grabación

`extract_features` aplica todo lo anterior y resume cada serie con
estadísticos:

| Familia | Series | Estadísticos | Total |
|---|---|---|---|
| MFCC | 13 | media, desvío, máx., mín. | 52 |
| Delta-MFCC | 13 | media, desvío | 26 |
| Centroide, ancho de banda, roll-off | 3 | media, desvío | 6 |
| Contraste | 7 bandas | media, desvío | 14 |
| Planitud, ZCR | 2 | media, desvío | 4 |
| RMS | 1 | media, desvío, máx. | 3 |
| | | | **105** |

In [ ]:
f = pd.Series(extract_features(cargar(EJEMPLOS["Soplo"])))
print(f"{len(f)} features para {EJEMPLOS['Soplo']}:")
print(f.round(3).to_string())

## 7. ¿Qué features dependen del volumen?

En la notebook 02 vimos que en pulmonares el volumen solo da AUC 0,78. Si una
feature está muy correlacionada con el volumen, es "otra forma" de medir el
volumen. Usamos las tablas de features de la tesis (`outputs/`) y calculamos
la correlación de cada feature con `rms_mean`.

In [ ]:
tablas = {"HS": pd.read_csv(OUT_DIR / "hs_features.csv"), "LS": pd.read_csv(OUT_DIR / "ls_features.csv")}
NO_FEATURES = ["file_id", "label", "label_binary"]

corr_volumen = {}
for subset, df in tablas.items():
    X = df.drop(columns=NO_FEATURES)
    corr_volumen[subset] = X.corrwith(X["rms_mean"], method="spearman").abs()
    print(f"\n{subset}: features más asociadas al volumen (|correlación de Spearman| con rms_mean)")
    print(corr_volumen[subset].drop("rms_mean").sort_values(ascending=False).head(8).round(2).to_string())
    print(f"-> {(corr_volumen[subset] > 0.8).sum()} de {X.shape[1]} features con |correlación| > 0,8")

Usamos la correlación de **Spearman** (por rangos) porque solo nos interesa si
"cuando sube el volumen, sube la feature", sin importar que la relación sea
lineal.

Que `mfcc_1_*` y `rms_*` estén arriba es esperable: miden energía. Lo
llamativo es que **otras features**, que en teoría describen la forma del
espectro (contraste, ancho de banda, otros MFCC), también siguen al volumen
de cerca. Son "features de volumen encubiertas". Normalizar la amplitud antes
de extraerlas (notebook 02) debería cortar esa dependencia.

## 8. ¿Qué features distinguen normal de patológico, una por una?

Para cada feature calculamos el AUC usándola sola: la probabilidad de que una
grabación patológica tenga un valor más alto que una normal. Como el sentido
no importa (más alto o más bajo), mostramos `max(AUC, 1 - AUC)`: 0,5 = no
distingue; 1 = separa perfecto.

**Advertencia:** esto es exploratorio. Está calculado con *todos* los datos,
así que **no** se puede usar para elegir features para el modelo: sería mirar
las respuestas antes del examen (fuga de información). Sirve para entender
qué tipo de información distingue las clases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (subset, df) in zip(axes, tablas.items()):
    X, y = df.drop(columns=NO_FEATURES), df["label_binary"]
    auc = X.apply(lambda col: roc_auc_score(y, col))
    fuerza = np.maximum(auc, 1 - auc).sort_values(ascending=False).head(15)
    colores = ["tab:red" if corr_volumen[subset][c] > 0.6 else "tab:blue" for c in fuerza.index]
    ax.barh(fuerza.index[::-1], fuerza.values[::-1], color=colores[::-1])
    ax.axvline(0.5, color="gray", ls=":")
    ax.set(xlim=(0.5, 1), title=f"{subset}: las 15 features que más distinguen, una por una",
           xlabel="max(AUC, 1 − AUC)")
fig.suptitle("En rojo: features asociadas al volumen (|correlación con rms_mean| > 0,6)")
fig.tight_layout()

Preguntas para mirar el gráfico:
- ¿Cuántas de las más discriminativas están asociadas al volumen (en rojo)?
  ¿Pasa más en pulmonares que en cardíacos?
- ¿Qué familias aparecen más arriba? ¿Tiene sentido con lo que sabés de cómo
  suena cada patología?

## 9. Tu turno

Elegí una feature y mirá su distribución en normales y patológicos. Probá con
alguna de las que aparecieron arriba y con alguna que no.

In [ ]:
SUBSET, FEATURE = "LS", "spectral_flatness_mean"

df = tablas[SUBSET]
fig, ax = plt.subplots(figsize=(6, 4))
for i, clase in enumerate([0, 1]):
    v = df.loc[df.label_binary == clase, FEATURE]
    ax.scatter(np.full(len(v), i) + np.random.uniform(-0.08, 0.08, len(v)), v, alpha=0.7)
ax.set(xticks=[0, 1], xticklabels=["Normal", "Patológico"], ylabel=FEATURE,
       title=f"{SUBSET} — AUC = {roc_auc_score(df.label_binary, df[FEATURE]):.2f}")
fig.tight_layout()

## Preguntas de control

1. ¿Por qué los modelos necesitan un vector de largo fijo? ¿Cuáles son los
   dos pasos para obtenerlo a partir de un audio?
2. ¿Qué información se pierde al resumir una serie con media, desvío, máximo y
   mínimo? ¿Qué patología te preocupa más por eso?
3. Explicá con tus palabras qué miden el centroide espectral, la planitud y la
   tasa de cruces por cero.
4. ¿Qué representa el MFCC 1? ¿Por qué eso conecta con el problema del volumen?
5. ¿Por qué cargar a 22.050 Hz desaprovecha el banco de filtros mel? Si
   pasáramos a 4000 Hz, ¿qué otro parámetro habría que cambiar y por qué?
6. ¿Por qué no podemos usar el gráfico de la sección 8 para elegir qué
   features darle al modelo?